# 7단계: 파인튜닝 어댑터 서빙 (조건 4·5) — 예측 생성 (Colab)

QLoRA 어댑터를 **조건 2·3이 쓴 것과 동일한 AWQ 베이스**에 얹어 예측을 생성한다.
베이스 가중치가 같아야 조건 2 → 4 차이가 파인튜닝 효과만 반영한다.

| 변형 | 프롬프트 파일 | 내용 |
|---|---|---|
| `local_ft` (조건 4) | `data/results/local_ft/prompts.jsonl` | RAG 없음 — 조건 2와 바이트 단위로 같은 프롬프트 |
| `local_ft_rag` (조건 5) | `data/results/local_rag_values_only/prompts.jsonl` | 3d 구성 — DB 값 주입 (few-shot 없음) |
| `local_ft_rag_v2` (조건 5′) | `data/results/local_rag_values_only_v2/prompts.jsonl` | 같은 구성, 값 매처 개선판 |

조건 5가 3d 프롬프트를 쓰는 이유는 RESULTS.md 참고: few-shot이 사는 건 Spider 문체이고 그건
파인튜닝이 이미 가중치에 넣으므로, 값 주입만이 파인튜닝과 겹치지 않는다.

조건 5′는 **어댑터를 다시 학습하지 않는다** — 조건 4·5와 같은 어댑터에 개선된 값 프롬프트만 얹는다.
값 주입 효과(+8, p=0.23)가 유의성에 못 미쳐, 효과 크기 자체를 키워 다시 재는 것이 목적이다.

**이 노트북의 첫 관문**: vLLM이 AWQ 베이스에 LoRA를 얹어주는지. 거부하면 아래 로드 셀에서 바로 실패한다.
그 경우 NF4 베이스로 서빙하고 조건 2 베이스라인만 다시 재면 된다(Colab 1회 + 채점 40초).

**어댑터는 학습(NF4)과 서빙(AWQ)의 양자화가 다르다** — 알려진 절충이며 RESULTS.md에 기록한다.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vllm
!pip uninstall -y -q torchaudio
!pip install -q -U torchvision
!pip install -q git+https://github.com/MumuKim0212/text2sql-rag-vs-finetune.git

## 어댑터 + 프롬프트 업로드

`VARIANT`를 맞추고, 해당 프롬프트 `prompts.jsonl`과 학습 노트북이 만든 `qwen_spider_qlora_adapter.zip`을 올린다.
한 세션에서 조건 4·5를 연달아 돌리려면 첫 변형이 끝난 뒤 `VARIANT`만 바꾸고 이 셀부터 다시 실행하면 된다 —
모델 로드 셀은 엔진이 이미 있으면 건너뛴다.

In [ ]:
import json
import zipfile
from pathlib import Path

# 조건 5는 "local_ft_rag" (프롬프트는 local_rag_values_only/prompts.jsonl)
# 조건 5'는 "local_ft_rag_v2" (프롬프트는 local_rag_values_only_v2/prompts.jsonl)
VARIANT = "local_ft"

PROMPTS_PATH = Path(f"{VARIANT}_prompts.jsonl")
OUT_PATH = Path(f"qwen_{VARIANT}_dev_predictions.jsonl")
ADAPTER_DIR = Path("qwen_spider_qlora_adapter")

if not PROMPTS_PATH.exists():
    from google.colab import files

    uploaded = files.upload()  # 해당 조건의 prompts.jsonl
    assert "prompts.jsonl" in uploaded, "prompts.jsonl을 업로드해야 함"
    Path("prompts.jsonl").rename(PROMPTS_PATH)

if not ADAPTER_DIR.exists():
    from google.colab import files

    uploaded = files.upload()  # qwen_spider_qlora_adapter.zip
    zip_name = next(n for n in uploaded if n.endswith(".zip"))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(ADAPTER_DIR)

# peft writes the weights next to adapter_config.json; find whichever level it landed on.
config = next(ADAPTER_DIR.rglob("adapter_config.json"))
ADAPTER_PATH = str(config.parent)
adapter_meta = json.loads(config.read_text(encoding="utf-8"))
print("prompts:", PROMPTS_PATH, "| out:", OUT_PATH)
print("adapter:", ADAPTER_PATH, "| r =", adapter_meta["r"], "| base =", adapter_meta.get("base_model_name_or_path"))

In [ ]:
import os

# Same Colab workarounds as conditions 2 and 3: vLLM's suppress_stdout() calls
# sys.stdout.fileno(), which ipykernel does not support, and it skips that call
# entirely when VLLM_LOGGING_LEVEL=DEBUG.
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"  # identical base to conditions 2 and 3

# Skip if an engine is already up: vLLM reserves gpu_memory_utilization of VRAM at
# construction, so building a second one fails before the first can be freed.
if "llm" in globals():
    print("engine already loaded, reusing:", MODEL_ID)
else:
    llm = LLM(
        model=MODEL_ID,
        quantization="awq",
        dtype="float16",
        gpu_memory_utilization=0.85,
        max_model_len=4096,
        enable_lora=True,
        max_lora_rank=64,  # the adapter was trained at r=64; vLLM defaults to 16
        max_loras=1,
    )
    print("loaded:", MODEL_ID)

## 어댑터가 실제로 적용되는지 검증

**이걸 건너뛰면 안 된다.** vLLM이 `lora_request`를 조용히 무시하면 조건 2와 동일한 출력이 나오고,
우리는 그걸 "파인튜닝 효과 없음"으로 잘못 기록하게 된다. 같은 프롬프트를 어댑터 있이/없이 돌려
출력이 달라지는지 확인한다.

In [ ]:
# Imported, not copied: conditions 1-3 all served this exact string, and a
# retyped copy could drift without anyone noticing.
from rag_text2sql.models.cloud import SYSTEM_PROMPT

prompts = [json.loads(line) for line in PROMPTS_PATH.open(encoding="utf-8") if line.strip()]
print(f"{len(prompts)} prompts, max {max(len(p['prompt']) for p in prompts)} chars")

sampling_params = SamplingParams(temperature=0.0, max_tokens=256)
lora = LoRARequest("spider", 1, ADAPTER_PATH)

probe = prompts[:8]
conversations = [
    [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": p["prompt"]}]
    for p in probe
]

with_lora = [o.outputs[0].text.strip() for o in llm.chat(conversations, sampling_params, lora_request=lora)]
without_lora = [o.outputs[0].text.strip() for o in llm.chat(conversations, sampling_params)]

changed = sum(1 for a, b in zip(with_lora, without_lora) if a != b)
print(f"{changed}/{len(probe)} of the probe prompts changed once the adapter was applied")
for i, (a, b) in enumerate(zip(without_lora, with_lora)):
    if a != b:
        print(f"\n[{i}] base : {a[:110]}")
        print(f"[{i}] lora : {b[:110]}")

assert changed > 0, "adapter had no effect on any probe prompt -- vLLM is ignoring lora_request"
print("\nadapter is being applied.")

## Spider dev 전체 생성

조건 2·3과 동일하게 32개 배치, 배치마다 append + flush로 재개 가능. 샘플링도 동일(`temperature=0.0`, `max_tokens=256`).
**모든 요청에 `lora_request`를 넘긴다** — 빼먹으면 베이스 모델 출력이 섞인다.

In [ ]:
BATCH_SIZE = 32

done = []
if OUT_PATH.exists():
    with OUT_PATH.open(encoding="utf-8") as f:
        done = [json.loads(line) for line in f if line.strip()]
start = len(done)
print(f"Resuming: {start} examples already done" if start else "Starting fresh")

with OUT_PATH.open("a", encoding="utf-8") as f:
    for batch_start in range(start, len(prompts), BATCH_SIZE):
        batch = prompts[batch_start : batch_start + BATCH_SIZE]
        conversations = [
            [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": ex["prompt"]}]
            for ex in batch
        ]
        outputs = llm.chat(conversations, sampling_params, lora_request=lora)
        for ex, out in zip(batch, outputs):
            record = {
                "question": ex["question"],
                "db_id": ex["db_id"],
                "gold_sql": ex["gold_sql"],
                "pred_sql": out.outputs[0].text.strip(),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        print(f"[{min(batch_start + BATCH_SIZE, len(prompts))}/{len(prompts)}]")

print("done:", OUT_PATH)

## 예측 파일 다운로드

받은 파일을 로컬 `data/results/{VARIANT}/`에 넣고 채점(1,034개에 약 40초):

```bash
uv run python scripts/score_predictions.py --predictions data/results/local_ft/qwen_local_ft_dev_predictions.jsonl --condition local_ft
uv run python scripts/score_predictions.py --predictions data/results/local_ft_rag/qwen_local_ft_rag_dev_predictions.jsonl --condition local_ft_rag
uv run python scripts/score_predictions.py --predictions data/results/local_ft_rag_v2/qwen_local_ft_rag_v2_dev_predictions.jsonl --condition local_ft_rag_v2
```

In [ ]:
from google.colab import files

files.download(str(OUT_PATH))